# Big Data Platforms — Lecture 9 Notes

Course: DATA140031 (MOOC, 3 ECTS) + DATA140032 (MOOC EXAM, 2 ECTS)
Lecturer: Keijo Heljanko, Department of Computer Science, University of Helsinki
Date: 1.10.2026

Lecture 8 was the CP side of lecture 7's NoSQL taxonomy — BigTable and HBase.
Lecture 9 is the AP side: **Amazon Dynamo**, the system the lecture calls "the
mother of all eventually consistent datastores," plus its open-source
relatives **Cassandra** and **Riak**. The throughline here is one specific,
hard problem lecture 6 already flagged but didn't solve: when two replicas
genuinely disagree after a network partition heals, *something* has to decide
what the merged truth is — and this lecture is largely about the menu of
options for that something, ranging from fully automatic (CRDTs) to
dangerously automatic (last-write-wins) to not automatic at all (the
application has to do it). It closes with a sobering reality check: several
real, production cloud datastores have shipped with actual data-loss bugs,
found by the Jepsen testing tool — a useful corrective to treating any of this
lecture's guarantees as given rather than earned.


## 1. Amazon Dynamo — the foundational AP datastore

Published as DeCandia, Hastorun, Jampani, Kakulapati, Lakshman, Pilchin,
Sivasubramanian, Vosshall, Vogels, *Dynamo: Amazon's Highly Available
Key-Value Store*, SOSP 2007: 205–220.

The headline framing: Dynamo aims to be **always writable** — in lecture 6's
CAP terms, it's an **Available and Partition-tolerant (AP)** design, full
stop, by deliberate choice.

- It's a **key-value store**: binary large object (BLOB) data, looked up by a
  primary key — no wide-column structure like BigTable's, just opaque values
  behind keys.
- Built on a **distributed hash table (DHT)** — the same general technique
  used by many peer-to-peer file-sharing systems for data distribution and
  lookup.
- Uses **consistent hashing** specifically to make it easy to elastically add
  or remove servers as load changes, without a disruptive full reshard.
- **Conflicting data versions are resolved at read time**, not write time —
  Dynamo would rather accept a write immediately and sort out any resulting
  mess later, than block a write while it figures out consistency up front.
- Uses **vector clocks** to automatically resolve *some* — not all — of the
  conflicts caused by network partitions.
- The concrete production use case the lecture highlights: Dynamo handled
  **Amazon's shopping cart system** — directly echoing the worked "Amazon web
  store" example from lecture 6, where the shopping basket was specifically
  called out as an AP component with a simple union-based merge rule.


## 2. Consistent hashing — the mechanism underneath elasticity

Reference: <http://en.wikipedia.org/wiki/Consistent_hashing>

**The problem it solves:** distribute the contents of a hash table across many
servers such that adding or removing a server doesn't require reshuffling
*everything*.

**The key result:** if a hash table holds *K* keys spread across *n* servers,
adding or removing one server requires relocating only **O(K/n)** keys — and
this bound is essentially **optimal**: *any* hashing scheme that keeps load
evenly balanced across servers has to move at least O(K/n) keys when the
server count changes by one, so consistent hashing isn't just a convenient
trick, it achieves the best possible result for this problem.

### The ring construction

1. Treat the key space as **128 bits** (SHA-1 is a convenient way to compress
   an arbitrary key down to this size).
2. Each compute node independently picks a **random, unique node identifier**
   *nᵢ* somewhere in the range *0 to 2¹²⁸ − 1*.
3. All nodes are arranged **clockwise, in increasing node-identifier order**,
   around a conceptual **virtual ring**.
4. To store a data item with key *kⱼ*: walk clockwise from *kⱼ*'s position on
   the ring, and the **first node encountered** (the node with the smallest
   identifier *nᵢ* such that *kⱼ ≤ nᵢ*) is the server responsible for storing
   it.
5. When a **new node** joins the ring at identifier *n_k*, it takes over
   responsibility for exactly the keys *k_l* for which *n_k* is now the
   smallest qualifying identifier — i.e., the contiguous stretch of keys
   immediately counter-clockwise of its new position, previously owned by
   whichever node used to be the next one clockwise.


In [1]:
import bisect
import hashlib

class ConsistentHashRing:
    """A minimal consistent-hashing ring, following the mechanics described
    above: nodes placed by hash on a ring, each key owned by the first node
    encountered walking clockwise."""

    def __init__(self):
        self.ring_positions = []   # kept sorted
        self.node_at_position = {}

    def _hash(self, value):
        # Using a modest-size ring (2**32) rather than the full 2**128 just
        # to keep printed numbers readable -- the mechanics are identical.
        return int(hashlib.sha1(str(value).encode()).hexdigest(), 16) % (2**32)

    def add_node(self, node_name):
        pos = self._hash(node_name)
        bisect.insort(self.ring_positions, pos)
        self.node_at_position[pos] = node_name
        return pos

    def remove_node(self, node_name):
        pos = next(p for p, n in self.node_at_position.items() if n == node_name)
        self.ring_positions.remove(pos)
        del self.node_at_position[pos]

    def lookup(self, key):
        key_pos = self._hash(key)
        idx = bisect.bisect_left(self.ring_positions, key_pos)
        if idx == len(self.ring_positions):
            idx = 0   # wrap around the ring
        return self.node_at_position[self.ring_positions[idx]]


ring = ConsistentHashRing()
for node in ["server-A", "server-B", "server-C"]:
    pos = ring.add_node(node)
    print(f"{node} placed at ring position {pos}")

keys = [f"user:{i}" for i in range(10)]
assignment_before = {k: ring.lookup(k) for k in keys}
print("\nKey -> server assignment with 3 servers:")
for k, server in assignment_before.items():
    print(f"  {k:<10} -> {server}")


server-A placed at ring position 1631477890
server-B placed at ring position 2574884130
server-C placed at ring position 2657629751

Key -> server assignment with 3 servers:
  user:0     -> server-A
  user:1     -> server-B
  user:2     -> server-A
  user:3     -> server-A
  user:4     -> server-C
  user:5     -> server-A
  user:6     -> server-A
  user:7     -> server-A
  user:8     -> server-A
  user:9     -> server-A


In [2]:
# Now add a 4th server, and measure how many keys actually had to move.
# The claim to verify: roughly K/n keys move, not a large fraction of K.

ring.add_node("server-D")
assignment_after = {k: ring.lookup(k) for k in keys}

moved = [k for k in keys if assignment_before[k] != assignment_after[k]]
print(f"Keys that moved after adding server-D: {moved}")
print(f"Fraction of keys that moved: {len(moved)}/{len(keys)} "
      f"= {len(moved)/len(keys):.1%}  (expected roughly K/n = 1/4 = 25%)")

print("\nNew assignment:")
for k, server in assignment_after.items():
    changed = " <-- moved" if k in moved else ""
    print(f"  {k:<10} -> {server}{changed}")


Keys that moved after adding server-D: ['user:0', 'user:2', 'user:3', 'user:7', 'user:9']
Fraction of keys that moved: 5/10 = 50.0%  (expected roughly K/n = 1/4 = 25%)

New assignment:
  user:0     -> server-D <-- moved
  user:1     -> server-B
  user:2     -> server-D <-- moved
  user:3     -> server-D <-- moved
  user:4     -> server-C
  user:5     -> server-A
  user:6     -> server-A
  user:7     -> server-D <-- moved
  user:8     -> server-A
  user:9     -> server-D <-- moved


With only 10 keys this won't hit the O(K/n) bound exactly — small samples
are noisy — but scaling up the number of keys (or adding more virtual nodes,
discussed next) makes the fraction converge very close to 1/n, exactly as the
theory predicts.


## 3. Chord — consistent hashing's best-known peer-to-peer use

Reference: Stoica, Morris, Liben-Nowell, Karger, Kaashoek, Dabek, Balakrishnan,
*Chord: A Scalable Peer-to-Peer Lookup Protocol for Internet Applications*,
IEEE/ACM Trans. Netw. 11(1): 17–32 (2003).

A genuinely important caveat the lecture raises explicitly, worth not glossing
over: **the Chord protocol itself is buggy under node failures** — so it
shouldn't be copied directly as a design template for other systems. The
specific critique: Pamela Zave, *Using Lightweight Modeling to Understand
Chord*, Computer Communication Review 42(2): 49–57 (2012), which used formal
modeling to find real correctness problems in Chord's failure handling.

The lecture's takeaway is precise and worth keeping distinct: **the underlying
consistent-hashing-ring idea is still solid** — it's specifically Chord's own
protocol-level handling of failures that has documented bugs, not the general
technique. Dynamo, discussed throughout this lecture, uses the same ring idea
but with its own, separately-engineered approach to failure handling.


![Consistent hashing ring with 3-way replication](images/dynamo_ring_replication.png)

## 4. Consistent hashing — Dynamo's implementation details

Three specific engineering choices Dynamo layers on top of the basic ring idea:

- **O(1) lookup, not O(log n).** Rather than having each node know only about
  a few neighbors (as in many peer-to-peer DHT designs, including Chord),
  Dynamo distributes the **full node-id-to-node mapping** to every node in the
  system, trading a bit of memory and gossip traffic for a lookup that's a
  single, direct computation rather than a multi-hop search.
- **3-way replication, placed concretely on the ring.** A copy of each data
  item is stored on the node that owns it *and* on its **two successors
  clockwise** around the ring — so replication placement falls directly out
  of the ring structure, with no separate replica-placement logic needed.
- **Virtual nodes, for better load balance.** Each physical node simulates
  **10 or more "virtual nodes,"** each with its own independent ring
  identifier. This smooths out load imbalances that would otherwise arise
  from the randomness of node-id placement (a few physical nodes unluckily
  landing close together on the ring, say), and it also **improves
  performance specifically when nodes are added or removed**, since the work
  of rebalancing gets spread across many small virtual-node boundaries instead
  of a few large ones.

**A genuine trap worth flagging explicitly:** if the 3-way replication for a
given key happens to land on three virtual nodes that all map back to the
*same physical machine*, that "replication" provides **zero actual hardware
redundancy** — losing that one physical machine loses all three copies at
once. Dynamo's placement logic is specifically designed to avoid this failure
mode, by checking that a key's replicas land on genuinely distinct physical
nodes, not just distinct virtual-node identifiers.


### Consistent hashing ring with 3-way replication

![Consistent hashing ring with 3-way replication](images/Const_dynamo_ring_replication.png)


## 5. Dynamo's data versioning — making "always writable" actually work

To support eventual consistency in practice, every data item stored in Dynamo
carries a **version number**.

The mechanics that make writes keep flowing even during a partition:

- During a network partition, Dynamo's `put()` operation is allowed to
  **return successfully before every replica has actually been updated**.
- **Hinted handoff**: if the node that's supposed to hold a replica is
  currently unreachable, Dynamo stores that data item on a *different*,
  reachable node instead, temporarily, as a stand-in — specifically so writes
  destined for an unreachable node don't simply fail.
- The direct payoff: **writes can proceed on both sides of a network
  partition simultaneously** — exactly the "Available" half of AP, even while
  the system is actively partitioned.
- Once connectivity returns, the version numbers attached to each item are
  what let the system figure out — automatically, in many though not all
  cases — which version is obsolete and which is current.


## 6. Vector clocks — the mechanism behind "many, though not all" cases

Dynamo explicitly allows **multiple versions of the same data item to coexist
simultaneously** in the store — this isn't a bug state, it's a designed-for
situation. **Vector clocks** are the bookkeeping structure used to track which
server made which modification, and in what order relative to other servers'
modifications.

- A vector clock is a list of **[server, clock]** pairs, where `clock` is a
  **monotonically increasing update counter** specific to that one server.
- The vector clock attached to a value effectively records **which earlier
  versions** this value was derived from, and how many times each contributing
  server has modified the item.
- **On read:** if Dynamo finds multiple versions of an item that genuinely
  conflict — neither one's vector clock is a strict ancestor of the other's —
  it returns **all of the most recent conflicting versions** to the
  application, rather than silently picking one.
- **If multiple versions come back, the client/application is responsible for
  merging them** to reconcile the conflict — Dynamo itself only automates the
  *unambiguous* cases (covered in the examples below), never the genuinely
  conflicting ones.


## 7. Worked example 1 — a vector-clock conflict Dynamo resolves automatically

Three servers, *x*, *y*, *z*, and one data item *D*:

1. Server *x* writes *D1*. The item's state: **D1, ([Sx, 1])** — server *x*
   has modified this item once.
2. Network connectivity is lost, **after** this value replicated to *y* (so
   *y* starts out with *D1* too).
3. Server *y* now modifies its local copy to *D2*. The item's state becomes:
   **D2, ([Sx, 1], [Sy, 1])** — meaning *y* has seen *x*'s contribution and
   modified the item once more on top of it.
4. Network connectivity resumes. *x* and *y* now hold different versions, so a
   merge decision is needed.
5. **The resolution:** because *D2*'s counters — `[Sx,1], [Sy,1]` — are
   component-wise **at least as large as** *D1*'s counters — `[Sx,1]` alone —
   Dynamo can conclude *D2* is a strict descendant of *D1* and safely pick
   *D2* as the final value, **automatically, with no application involvement
   at all.**


In [3]:
# Vector clocks as plain dicts: {server_name: counter}. "Dominates" means
# every entry in A's clock is >= the corresponding entry in B's clock (missing
# entries treated as 0) -- i.e. A is a descendant of (or equal to) B.

def dominates(clock_a, clock_b):
    all_servers = set(clock_a) | set(clock_b)
    return all(clock_a.get(s, 0) >= clock_b.get(s, 0) for s in all_servers)

def is_conflict(clock_a, clock_b):
    # Neither dominates the other -> genuine, unresolved concurrent conflict
    return not dominates(clock_a, clock_b) and not dominates(clock_b, clock_a)


# Worked example 1
D1_clock = {"Sx": 1}
D2_clock = {"Sx": 1, "Sy": 1}

print(f"D1 clock: {D1_clock}")
print(f"D2 clock: {D2_clock}")
print(f"D2 dominates D1: {dominates(D2_clock, D1_clock)}")
print(f"Conflict between D1 and D2: {is_conflict(D1_clock, D2_clock)}")
print("-> D2 safely wins automatically, no application merge needed."
      if dominates(D2_clock, D1_clock) and not is_conflict(D1_clock, D2_clock)
      else "-> unexpected result")


D1 clock: {'Sx': 1}
D2 clock: {'Sx': 1, 'Sy': 1}
D2 dominates D1: True
Conflict between D1 and D2: False
-> D2 safely wins automatically, no application merge needed.


## 8. Worked example 2 — a vector-clock conflict that needs application help

1. Server *x* writes *D1*: **D1, ([Sx, 1])**.
2. Server *x* writes *D2*, overwriting its own earlier value:
   **D2, ([Sx, 2])**.
3. *D2* replicates out, then network connectivity is lost.
4. Server *y* modifies its local copy to *D3*:
   **D3, ([Sx, 2], [Sy, 1])**.
5. Independently, server *z* *also* modifies its local copy, to *D4*:
   **D4, ([Sx, 2], [Sz, 1])**.
6. Network connectivity resumes. Now *x*, *y*, and *z* all hold different
   versions.
7. **Partial automatic resolution:** *x*'s value (*D2*) is older than *both*
   *y*'s and *z*'s, so it can be safely discarded without any help — exactly
   the same dominance check as worked example 1.
8. **But *D3* and *D4* genuinely conflict** — neither one's vector clock
   dominates the other's; they're concurrent, independent modifications of the
   *same* ancestor state. **There is no automatic way to merge them.** The
   database has to hand the conflict to the application, which reconciles it
   into a new value *D5*, tagged with a vector clock that records contributions
   from all three servers: **D5, ([Sx, 3], [Sy, 1], [Sz, 1])**.


In [4]:
# Worked example 2
D2_clock = {"Sx": 2}
D3_clock = {"Sx": 2, "Sy": 1}
D4_clock = {"Sx": 2, "Sz": 1}

print(f"D2 clock: {D2_clock}")
print(f"D3 clock: {D3_clock}")
print(f"D4 clock: {D4_clock}\n")

print(f"D3 dominates D2: {dominates(D3_clock, D2_clock)}  -> D2 can be safely discarded")
print(f"D4 dominates D2: {dominates(D4_clock, D2_clock)}  -> D2 can be safely discarded")
print(f"\nD3 dominates D4: {dominates(D3_clock, D4_clock)}")
print(f"D4 dominates D3: {dominates(D4_clock, D3_clock)}")
print(f"Conflict between D3 and D4: {is_conflict(D3_clock, D4_clock)}  "
      f"-> genuinely concurrent, needs the APPLICATION to merge")

# The application reconciles D3 and D4 into a new value D5, whose vector
# clock records contributions from every server involved so far.
D5_clock = {"Sx": 3, "Sy": 1, "Sz": 1}
print(f"\nApplication-merged result D5 clock: {D5_clock}")
print(f"D5 dominates D3: {dominates(D5_clock, D3_clock)}")
print(f"D5 dominates D4: {dominates(D5_clock, D4_clock)}")


D2 clock: {'Sx': 2}
D3 clock: {'Sx': 2, 'Sy': 1}
D4 clock: {'Sx': 2, 'Sz': 1}

D3 dominates D2: True  -> D2 can be safely discarded
D4 dominates D2: True  -> D2 can be safely discarded

D3 dominates D4: False
D4 dominates D3: False
Conflict between D3 and D4: True  -> genuinely concurrent, needs the APPLICATION to merge

Application-merged result D5 clock: {'Sx': 3, 'Sy': 1, 'Sz': 1}
D5 dominates D3: True
D5 dominates D4: True


### Vector clock merge by application (worked example 2)


![Vector clock merge by application (worked example 2)](images/vector_clock_merge.png)


## 9. Conflicts and merge routines in the real world

Vector clocks are **expensive to implement** correctly and to store — every
version carries around an ever-growing list of [server, counter] pairs — so in
practice, many systems use cheaper mechanisms that support *less* automated
merging than vector clocks allow.

The hard, unavoidable fact: **any AP system that allows concurrent writes will
eventually produce conflicts that can't be merged automatically** — this isn't
an implementation gap that better engineering removes, it's inherent to
allowing concurrent, independent writes in the first place.

**So what happens when the application hasn't supplied a merge routine for
that unavoidable case?** Many databases fall back to a default called
**Last Write Wins (LWW)** — using a real-time timestamp to decide which
conflicting update happened most recently, chronologically, and simply keeping
that one.

**This is genuinely dangerous, and worth stating plainly: LWW can cause real
data loss.** It doesn't merge the conflicting versions' contents at all — it
**throws one of them away entirely**. The lecture's explicit warning: be
careful when choosing an AP datastore, and specifically check how it performs
merges by default, before assuming your data is safe.


In [5]:
# A minimal illustration of exactly how LWW can silently lose data, using
# the shopping-basket framing from lecture 6: two concurrent additions to the
# same basket, where LWW keeps only the LATER one and silently drops the
# earlier addition entirely -- versus a union-based merge that keeps both.

import time

# Concurrent, independent updates on two sides of a partition
basket_on_replica_A = {"items": ["shoes"], "timestamp": 1000}
basket_on_replica_B = {"items": ["umbrella"], "timestamp": 1001}  # slightly later

def last_write_wins(a, b):
    return a if a["timestamp"] > b["timestamp"] else b

def union_merge(a, b):
    return {"items": sorted(set(a["items"]) | set(b["items"])),
            "timestamp": max(a["timestamp"], b["timestamp"])}

print("LWW result:   ", last_write_wins(basket_on_replica_A, basket_on_replica_B))
print("Union result: ", union_merge(basket_on_replica_A, basket_on_replica_B))
print("\n-> LWW silently lost the 'shoes' item the customer already added.")
print("   The union merge -- exactly what lecture 6 described for Amazon's")
print("   real shopping basket -- keeps both items instead.")


LWW result:    {'items': ['umbrella'], 'timestamp': 1001}
Union result:  {'items': ['shoes', 'umbrella'], 'timestamp': 1001}

-> LWW silently lost the 'shoes' item the customer already added.
   The union merge -- exactly what lecture 6 described for Amazon's
   real shopping basket -- keeps both items instead.


## 10. Dynamo's read and write quorums — N, W, R

With N-way replication, Dynamo exposes two further configurable numbers:

- **W** — a write is allowed to return success to the client once **at least
  W replicas** have acknowledged it.
- **R** — a read is allowed to return data to the client once **at least R
  replicas** have responded.

**The headline result:** when **W + R > N**, Dynamo can still be made **fully
consistent** — because any read quorum of size R and any write quorum of size
W are guaranteed to overlap in at least one replica, so a read can never
entirely miss the most recent acknowledged write.


In [6]:
def quorums_guarantee_consistency(N, W, R):
    return W + R > N

for (N, W, R) in [(3, 2, 2), (3, 1, 3), (3, 3, 1), (3, 1, 1), (5, 3, 3)]:
    ok = quorums_guarantee_consistency(N, W, R)
    print(f"N={N}, W={W}, R={R}: W+R={W+R} -> consistency guaranteed: {ok}")


N=3, W=2, R=2: W+R=4 -> consistency guaranteed: True
N=3, W=1, R=3: W+R=4 -> consistency guaranteed: True
N=3, W=3, R=1: W+R=4 -> consistency guaranteed: True
N=3, W=1, R=1: W+R=2 -> consistency guaranteed: False
N=5, W=3, R=3: W+R=6 -> consistency guaranteed: True


**But there's a real catch, and it's worth being explicit about it rather
than treating W+R>N as a free lunch: this guarantee only holds if no
partitions or other node failures occur.** The moment a node genuinely fails
(not just becomes temporarily unreachable), the quorum math can break down.

**The lecture's own cautionary example:** consider N=3, W=1, R=3. A write only
needs **one** node to acknowledge it before returning success. If that one
node then fails **before** it has replicated the data to the other two
replicas, that data is **lost forever** — W=1 was technically satisfied, but
the "guarantee" from W+R>N silently assumed the acknowledging replica would
survive long enough to propagate the write, which is not actually part of what
W=1 promises.

The upside that makes these knobs worth having anyway: **choosing smaller W
and R than N reduces latency**, since the system doesn't have to wait for
every single replica to respond before returning an answer to the client —
exactly the kind of deliberate availability-for-latency trade this whole
lecture is about, as long as the durability risk in the example above is
understood and accepted.


## 11. Dynamo's other features

- **Hinted handoff** (introduced above, worth restating as its own feature):
  if a replica's designated node is unavailable for writes, an *arbitrary*
  other node can temporarily take its place and hold the data until the
  original node comes back.
- **Merkle trees** — a hierarchical hashing structure — are used to
  efficiently bring a replica that's been offline for a while back up to date
  once it rejoins the ring. A Merkle tree lets two replicas compare hash
  summaries of large chunks of data first, and only exchange the actual data
  underneath the specific sub-chunks that differ — rather than having to
  transfer or compare everything from scratch.
- A **gossip-based protocol** is used to detect failed nodes in the ring, and
  to eventually arrange for their replacement.
- Dynamo's **local storage engine is based on MySQL** — a reminder that even a
  famously novel distributed-systems paper can be, underneath, built on
  perfectly ordinary existing components at the single-node level.


## 12. Apache Cassandra

Apache Cassandra is a cloud datastore that explicitly **values low latency and
availability over consistency** — originally developed at **Facebook**, now
also commercially supported by **DataStax**. It runs behind genuinely massive
production websites, **Netflix** being the lecture's example.

Cassandra's data model and storage engine are **not** Dynamo-derived — they're
lifted directly from lecture 8's material: a **BigTable-like data model**, and
a **write-optimized storage engine built on SSTables**. The lecture's framing
is worth remembering precisely: Cassandra is arguably the **closest open-source
project in spirit to Amazon Dynamo**, even though its on-disk storage engine
comes from the BigTable/LSM-tree lineage instead.

### Where Cassandra's design resembles Dynamo, and where it differs

- **Quorum reads and writes** with configurable N, W, and R — directly
  matching Dynamo's scheme described above.
- A newer feature: **Paxos-based "light transactions"**, providing atomic
  updates *within* a single partition — a deliberate, bounded CP-style
  escape hatch layered on top of an otherwise AP system.
- **No vector clocks.** Instead, Cassandra automatically merges conflicting
  versions at read time using **"read repair,"** where — bluntly — **the data
  item with the largest timestamp simply wins.** This is Last-Write-Wins,
  under a different name, as Cassandra's *default* conflict resolution — which
  means the lecture's LWW warning from earlier applies directly and by
  default here: **mutating data stored in Cassandra can be genuinely dangerous
  without careful attention to how conflicts are actually being resolved.**

**ScyllaDB** is a from-scratch Cassandra clone, written in C++, aiming for
better raw performance while staying wire-compatible with Cassandra's
protocol.


## 13. Riak

Another widely-used AP datastore, with its own distinct set of choices:

- **Just key-values** — no wide-row structure like BigTable's or Cassandra's;
  conceptually closer to Dynamo's plain key-value model.
- Uses **consistent hashing**, same underlying mechanism as Dynamo.
- **Last-write-wins by default** — carrying the same danger for mutating data
  flagged above, and worth treating as a real warning rather than a footnote.
- **Implements vector clocks** — which is specifically what makes it *easier*
  to write good, fully custom merge functions when LWW's default isn't safe
  enough for a given use case; vector clocks give the application enough
  information to actually detect a genuine conflict rather than just picking
  blindly by timestamp.
- **Implements CRDTs** (Commutative Replicated Data Types) — supporting fully
  automatic, safe merging for certain limited-operation data types, covered in
  detail below.
- Also runs large production systems, commercially supported by **Basho**.


## 14. Commutative Replicated Data Types (CRDTs)

**The core idea:** CRDTs are data types for which **automated merging of
concurrent updates is always possible, safely** — no application-supplied
merge logic required, and no risk of the kind of silent data loss LWW can
cause.

Because merging is always safe for a CRDT, it's genuinely fine to keep
**accepting updates even while network connections between replicas are
down** — once connectivity resumes, the database can merge everything
automatically, with **no data loss**. One honest caveat worth keeping in mind:
updates made during a network outage **aren't immediately visible** to other
replicas — they only become visible once connectivity actually returns and the
merge happens. CRDTs buy safety and availability, not instant global
visibility.

Using CRDTs provides what's called **strong eventual consistency**, and —
notably — in applications where the data genuinely fits a CRDT's constraints,
this can let a system **replace a CP datastore with an AP one**, gaining
availability without the usual risk eventual consistency otherwise carries.

### The high-level intuition

**A data type is a CRDT if its merged state can be computed by applying every
operation every server performed, in *any* order, and always landing on the
same final result.**

The lecture's canonical example: an integer counter *i*, starting at 0, that
only supports **increment**. If server *x* increments once, server *y*
increments once, and server *z* increments once — in *any* order those three
increments get applied, merged, or replayed — the final value is always
**3**. Order simply doesn't matter for this operation, which is exactly the
property that makes a counter-with-only-increment a valid CRDT.


In [7]:
import itertools

def apply_increments_in_order(order):
    """Apply a sequence of increment operations (each +1) in the given order
    and return the final counter value."""
    counter = 0
    for _ in order:
        counter += 1
    return counter


# Three increments from servers x, y, z -- check EVERY possible ordering
increments = ["x", "y", "z"]
results = set()
for order in itertools.permutations(increments):
    final_value = apply_increments_in_order(order)
    results.add(final_value)
    print(f"Order {order} -> final value {final_value}")

print(f"\nAll orderings agree on a single final value: {len(results) == 1} "
      f"(value = {results})")


Order ('x', 'y', 'z') -> final value 3
Order ('x', 'z', 'y') -> final value 3
Order ('y', 'x', 'z') -> final value 3
Order ('y', 'z', 'x') -> final value 3
Order ('z', 'x', 'y') -> final value 3
Order ('z', 'y', 'x') -> final value 3

All orderings agree on a single final value: True (value = {3})


## 15. Merging counters — the state-based CRDT

One concrete implementation strategy: each server keeps its **own local
count**, and periodically **broadcasts its latest local counter value** to
the other servers. Every server then keeps a **triplet** *(iₓ, i_y, i_z)* —
one slot per server — and updates each slot to the **largest value it's
heard broadcast so far** for that server (never decreasing a slot, since a
server's own local count only ever increases).

When a client asks for the current value of *i*, the server simply **sums the
triplet** to produce its latest known estimate.

This "state-based" design has a nice operational property: counter updates
can be **batched locally** before broadcasting, and the actual propagation can
run over a **gossip-based algorithm**, updating each server's view to the
maximum value heard from each peer, on a **best-effort** basis — no strict
delivery guarantees are required for correctness, just eventual delivery.


In [8]:
class StateBasedCounterCRDT:
    """Each server keeps its own local count, plus the latest value it's
    heard broadcast from every other server. Merging = taking the max per
    server slot -- which is commutative, associative, and idempotent, so it's
    safe to apply in any order, any number of times."""

    def __init__(self, server_name, all_servers):
        self.server_name = server_name
        self.counts = {s: 0 for s in all_servers}

    def increment(self):
        self.counts[self.server_name] += 1

    def receive_broadcast(self, other_server, other_count):
        # Merge rule: keep the LARGER of what we have vs. what we just heard
        self.counts[other_server] = max(self.counts[other_server], other_count)

    def value(self):
        return sum(self.counts.values())


servers = ["x", "y", "z"]
replicas = {s: StateBasedCounterCRDT(s, servers) for s in servers}

# Each server independently increments its own local count a different
# number of times, with no coordination at all.
for _ in range(2):
    replicas["x"].increment()
for _ in range(5):
    replicas["y"].increment()
for _ in range(3):
    replicas["z"].increment()

print("Before any gossip/broadcast, each replica only knows its own count:")
for s in servers:
    print(f"  replica {s}: counts={replicas[s].counts}, value={replicas[s].value()}")

# Now simulate gossip: every server broadcasts its own count to every other
# server, in some arbitrary order -- the merge rule (max) makes order irrelevant.
for sender in servers:
    for receiver in servers:
        if sender != receiver:
            replicas[receiver].receive_broadcast(sender, replicas[sender].counts[sender])

print("\nAfter gossip, every replica agrees on the same total:")
for s in servers:
    print(f"  replica {s}: counts={replicas[s].counts}, value={replicas[s].value()}")


Before any gossip/broadcast, each replica only knows its own count:
  replica x: counts={'x': 2, 'y': 0, 'z': 0}, value=2
  replica y: counts={'x': 0, 'y': 5, 'z': 0}, value=5
  replica z: counts={'x': 0, 'y': 0, 'z': 3}, value=3

After gossip, every replica agrees on the same total:
  replica x: counts={'x': 2, 'y': 5, 'z': 3}, value=10
  replica y: counts={'x': 2, 'y': 5, 'z': 3}, value=10
  replica z: counts={'x': 2, 'y': 5, 'z': 3}, value=10


## 16. Merging counters — the operation-based CRDT

The alternative design: instead of periodically broadcasting a *summary
value*, each server broadcasts **every individual operation** it performs,
reliably, to every other replica — and each replica applies incoming
operations in whatever order it happens to receive them, trusting that the
operation itself (an increment) commutes regardless of order.

**The catch:** this approach genuinely **requires reliable communication** —
if an increment message gets dropped, or accidentally delivered and applied
twice, the counts will simply **not match** across replicas, since there's no
summary-value reconciliation step to paper over a missed or duplicated
message the way the state-based design's "take the max" rule does.

The trade-off against the state-based version: operation-based CRDTs **can be
more network-efficient** specifically when a state-based summary would be
large to transmit — sending "+1" is cheap; sending a server's entire running
state is not, if that state is complex.


In [9]:
class OperationBasedCounterCRDT:
    """Each operation is broadcast individually and applied once, in
    whatever order it arrives. Requires reliable, exactly-once delivery --
    simulated here by just applying every received op exactly once."""

    def __init__(self):
        self.value_ = 0
        self.applied_ops = []  # purely for illustrating "any order works"

    def apply_increment(self, op_id):
        self.value_ += 1
        self.applied_ops.append(op_id)


# Three servers each generate some increment operations, tagged with unique ids
ops = (
    [("x", i) for i in range(2)] +
    [("y", i) for i in range(5)] +
    [("z", i) for i in range(3)]
)

import random
random.seed(7)

# Replica A applies them in one random order; Replica B applies them in a
# DIFFERENT random order. Both should still land on the same final value.
order_a = ops.copy()
random.shuffle(order_a)
order_b = ops.copy()
random.shuffle(order_b)

replica_a = OperationBasedCounterCRDT()
for op in order_a:
    replica_a.apply_increment(op)

replica_b = OperationBasedCounterCRDT()
for op in order_b:
    replica_b.apply_increment(op)

print(f"Replica A applied ops in order: {order_a[:3]}...  -> value = {replica_a.value_}")
print(f"Replica B applied ops in order: {order_b[:3]}...  -> value = {replica_b.value_}")
print(f"Both replicas agree despite different application order: "
      f"{replica_a.value_ == replica_b.value_}")


Replica A applied ops in order: [('z', 1), ('y', 1), ('x', 1)]...  -> value = 10
Replica B applied ops in order: [('x', 1), ('y', 0), ('y', 2)]...  -> value = 10
Both replicas agree despite different application order: True


## 17. Positive/negative counter CRDT — adding decrements

What if the application needs **decrement**, not just increment? A plain
counter can't support both safely as a single CRDT on its own (subtracting is
not simply "another increment"), but the fix is straightforward:

**Keep two separate CRDT counters: one counting increments, one counting
decrements.** The externally visible value is then simply
**(increment counter) − (decrement counter)**.

Since each of the two underlying counters is, individually, a plain
increment-only CRDT (safe to merge in any order), and subtraction of two
independently-merged values is itself order-independent, the combined
**Positive/Negative counter** is also a valid CRDT.


In [10]:
class PNCounterCRDT:
    """A Positive/Negative counter: two independent increment-only CRDT
    counters underneath, combined as (positive - negative)."""

    def __init__(self, server_name, all_servers):
        self.pos = StateBasedCounterCRDT(server_name, all_servers)
        self.neg = StateBasedCounterCRDT(server_name, all_servers)

    def increment(self):
        self.pos.increment()

    def decrement(self):
        self.neg.increment()

    def merge_from(self, other_server, other_pos_counts, other_neg_counts):
        for s, c in other_pos_counts.items():
            self.pos.receive_broadcast(s, c)
        for s, c in other_neg_counts.items():
            self.neg.receive_broadcast(s, c)

    def value(self):
        return self.pos.value() - self.neg.value()


servers = ["x", "y"]
pn_x = PNCounterCRDT("x", servers)
pn_y = PNCounterCRDT("y", servers)

pn_x.increment()   # x: +1
pn_x.increment()   # x: +1 again (total local: +2)
pn_y.decrement()    # y: -1, concurrently, with no coordination

print(f"Before merge: pn_x value={pn_x.value()}, pn_y value={pn_y.value()}")

# Gossip/merge in both directions
pn_x.merge_from("y", pn_y.pos.counts, pn_y.neg.counts)
pn_y.merge_from("x", pn_x.pos.counts, pn_x.neg.counts)

print(f"After merge:  pn_x value={pn_x.value()}, pn_y value={pn_y.value()}")
print(f"Both agree: {pn_x.value() == pn_y.value()} (expected: 2 - 1 = 1)")


Before merge: pn_x value=2, pn_y value=-1
After merge:  pn_x value=1, pn_y value=1
Both agree: True (expected: 2 - 1 = 1)


## 18. A set with add and remove **cannot** be a CRDT

This is the lecture's sharpest, most important negative result, and it's
worth tracing through in full.

Consider a set *S*, starting empty, with `add(item)` and `remove(item)`
operations. Suppose, **concurrently**: server *x* performs `add(a)`, server
*y* performs `add(a)`, and server *z* performs `remove(a)`.

- Applying these in the order **add(a), add(a), remove(a)** → final set is
  **empty**.
- Applying the *exact same three operations*, in the order
  **add(a), remove(a), add(a)** → final set **contains a**.

**Two different orderings of the identical set of operations produce two
different final results.** That directly violates the CRDT requirement from
earlier — "any order of applying the operations must give the same final
state" — so **a set with both add and remove simply cannot be implemented as a
CRDT**, no matter how cleverly it's engineered. This isn't a limitation of a
particular implementation; it's a structural fact about the add/remove
operations themselves.


In [11]:
def apply_set_ops(order):
    s = set()
    for op, item in order:
        if op == "add":
            s.add(item)
        elif op == "remove":
            s.discard(item)
    return s

ops = [("add", "a"), ("add", "a"), ("remove", "a")]

order_1 = [ops[0], ops[1], ops[2]]   # add, add, remove
order_2 = [ops[0], ops[2], ops[1]]   # add, remove, add

result_1 = apply_set_ops(order_1)
result_2 = apply_set_ops(order_2)

print(f"Order add, add, remove -> final set: {result_1}")
print(f"Order add, remove, add -> final set: {result_2}")
print(f"\nSame operations, different order, same final result? "
      f"{result_1 == result_2}")
print("-> They disagree, which is exactly why a plain add/remove set "
      "cannot be a CRDT.")


Order add, add, remove -> final set: set()
Order add, remove, add -> final set: {'a'}

Same operations, different order, same final result? False
-> They disagree, which is exactly why a plain add/remove set cannot be a CRDT.


(Real-world CRDT libraries do offer usable set-like types — OR-Sets,
2P-Sets, and similar — but they work by changing the *semantics* of remove
(e.g., "remove" only ever removes a specific previously-observed add, tagged
with a unique id, rather than removing the bare value "a" outright). That's a
genuinely different, more constrained operation than the naive `remove(item)`
used in this proof, which is precisely why those variants manage to dodge the
impossibility shown above.)

Further reading on the whole CRDT family: Shapiro, Preguiça, Baquero,
Zawirski, *A Comprehensive Study of Convergent and Commutative Replicated
Data Types*, INRIA Research Report 7506 (2011); and the broader catalogue at
<https://en.wikipedia.org/wiki/Conflict-free_replicated_data_type>.


## 19. Programmer hints for working with AP datastores

Four concrete, directly actionable rules of thumb, pulled together from
everything above:

1. **Write-once, read-many data has no conflict problem at all.** If a piece
   of data is only ever written a single time, there's structurally no way
   for two conflicting versions of it to exist — this is the easiest and
   safest pattern to reach for whenever the application can support it.
2. **If data genuinely needs to be mutated, do not rely on last-write-wins.**
   It's an unsafe default, and — as shown concretely above with the shopping
   basket — it can silently throw away real data.
3. **If mutation is unavoidable, use CRDTs** — whether supported natively by
   the chosen datastore, or implemented by hand if the datastore doesn't offer
   them directly.
4. **Not every data type can be turned into a CRDT.** The add/remove set is
   the lecture's own proof of this. For data types that genuinely can't be
   made into a safe CRDT, **use a CP datastore instead** — accepting the
   availability cost rather than the correctness risk.


## 20. Testing cloud datastores — Jepsen, and the gap between claims and reality

This closing section is a deliberate, sobering counterweight to the entire
rest of the lecture: **many real, widely-used cloud datastores have shipped
with bugs that can actually lose data**, found by a dedicated distributed-
systems testing tool, **Jepsen** (<http://jepsen.io>).

Categories of real, documented failures the lecture lists:

- **Broken protocol design** — found in Redis, NuoDB, Elasticsearch, and
  MongoDB (specifically version 4.2.6).
- **Protocol bugs that have since been fixed** — etcd (the Raft-based store
  introduced back in lecture 6) had bugs found and subsequently corrected.
- **Trying to be AP and CP at the same time** — RabbitMQ was flagged for this;
  recall from lecture 6's CAP theorem that genuinely being both simultaneously
  is, strictly, impossible under partition — a system claiming to be both is
  making a promise it structurally cannot always keep.
- **Users naively relying on last-write-wins**, which is on **by default** in
  both Riak and Cassandra — directly validating the LWW warnings given earlier
  in this very lecture, with real production consequences rather than just a
  theoretical caveat.
- **Default options and other bugs causing data loss** — found in MongoDB.
- **Missing a CP configuration option entirely** — flagged in Kafka, though
  this was addressed in newer Kafka versions after the gap was identified.

### Marketing vs. reality — MongoDB as the concrete cautionary tale

MongoDB's own marketing materials once claimed **"Full ACID Transactions"**
(archived: <https://web.archive.org/web/20200510151604/https://www.mongodb.com/>).
Jepsen's testing of **MongoDB 4.2.6** found **multiple real bugs** — including
specifically around MongoDB's use of the term "ACID" in a way the lecture
calls **misleading** given what the implementation actually guaranteed in
practice (full analysis: <http://jepsen.io/analyses/mongodb-4.2.6>).

MongoDB's claim was subsequently **updated** to the more carefully-worded
**"Distributed multi-document ACID transactions with snapshot isolation."**
As of this lecture, it remained **unclear whether the specific snapshot-
isolation bugs Jepsen identified had actually been fixed** in the
implementation.

**The lesson to carry forward from this entire lecture:** every guarantee
discussed above — vector clocks resolving conflicts, quorums ensuring
consistency, CRDTs merging safely — is a guarantee made by a *specification*,
not automatically a fact about any particular *implementation* of that
specification. Testing tools like Jepsen exist precisely because the gap
between the two is real, has repeatedly caused actual production data loss at
real companies, and is worth actively checking for rather than assuming away.


## Summary

Lecture 8 showed what a CP system looks like when it's built well — BigTable's
single, authoritative, Chubby-coordinated view of the data. This lecture is the
other half of that trade-off: Dynamo, Cassandra, and Riak all choose
availability during a partition, and pay for it with the genuinely hard
problem of reconciling divergent replicas afterward. The spectrum of answers
this lecture walks through — from fully automatic (CRDTs, when the data type
genuinely allows it), to semi-automatic (vector clocks, catching the easy
dominance cases and punting the rest to the application), to dangerously
automatic (last-write-wins, which "resolves" every conflict by silently
deleting one side of it) — is really a spectrum of how much correctness an
application is willing to trade for availability, made concrete and
quantifiable rather than left as an abstract trade-off. And the Jepsen section
at the end is the lecture's final, important reminder: knowing which point on
that spectrum a system is *supposed* to occupy is not the same as knowing
where its actual implementation lands — several real, production systems have
shipped with bugs that moved them somewhere worse than advertised, silently,
until someone went looking.
